### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

    -Tracking agent behavior with logging, analytics, and debugging.
    -Transforming prompts, tool selection, and output formatting.
    -Adding retries, fallbacks, and early termination logic.
    -Applying rate limits, guardrails, and PII detection.


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

1. Long-running conversations that exceed context windows.
2. Multi-turn dialogues with extensive history.
3. Applications where preserving full conversation context matters.

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message based summarization

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger = ("messages", 10),
            keep=("messages",4)
        )
    ]
)

In [7]:
# Run with the thread id

config={"configurable":{"thread_id":"test-1"}}

In [12]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response["messages"])}")

Messages: {'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to receive answers to basic arithmetic operations.\n\n## SUMMARY\nThe conversation history includes a series of simple math problems with their respective solutions: 2+2=4, 10*5=50, 100/4=25, 15-7=8, and 3*3. The user has repeated some math problems, with the answers consistently being provided.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe user has asked multiple math problems, and the next step is to await a new math problem from the user or assist with a different math question if needed.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='e8ad4436-e645-4414-941d-c4468f534808'), AIMessage(content="3 * 3 = 9. Simple multiplication. What's the next math problem you'd like to solve?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 300, 'total_tokens': 324, 'completion_tim